## 1. Loading the libraries

In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage
from pydantic import BaseModel, Field
from typing import List
load_dotenv()


True

In [26]:
class WindowChatMessageHistory(BaseChatMessageHistory,BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list)
    k:int = Field(default=6)

    def add_messages(self, messages: List[BaseMessage]):
        self.messages.extend(messages)
        if len(self.messages) > self.k:
            dropped = len(self.messages) - self.k
            self.messages = self.messages[-self.k:]
            print(f"Dropped {dropped} messages from history")
            print(f"Current messages: {self.messages}")

    def clear(self):
        self.messages = []


In [27]:
memory = WindowChatMessageHistory()

In [40]:
memory.add_messages([HumanMessage(content='what is AI')])

Dropped 1 messages from history
Current messages: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is AI', additional_kwargs={}, response_metadata={})]


In [41]:
memory.messages

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is AI', additional_kwargs={}, response_metadata={})]

In [42]:

WINDOW_K = 6 
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])


In [43]:
window_store={}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] =  WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [46]:
get_session_history('rbyte123')

WindowChatMessageHistory(messages=[], k=6)

In [47]:
window_store

{'rahul123': WindowChatMessageHistory(messages=[], k=6),
 'rbyte123': WindowChatMessageHistory(messages=[], k=6)}

In [51]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [52]:
chain = prompt | llm | StrOutputParser()

In [ ]:
window_assistant = RunnableWithMessageHistory(
chain,
get_session_history,
input_messages_key="input",
history_messages_key="history"
)


In [ ]:
cfg = {"configurable": {"session_id": "window_john"}}

In [ ]:
output = window_assistant.invoke({"input": "I have a billing issue with my last invoice."},
   config = cfg
)

In [58]:
print(output)

Classification: Billing  
Priority: High  

I understand that you have a billing issue with your last invoice. Could you please provide more details about the problem? This will help me assist you better.


In [67]:
window_store['window_john'].messages

[HumanMessage(content='I have a billing issue with my last invoice.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classification: Billing  \nPriority: High  \n\nI understand that you have a billing issue with your last invoice. Could you please provide more details about the problem? This will help me assist you better.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What category is his complaint?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The complaint is categorized as:  \nClassification: Billing  \nPriority: High  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the SLA for billing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The standard Service Level Agreement (SLA) for billing inquiries typically ranges from 24 to 48 hours for initial response. However, response times can vary based on 

In [65]:
output = window_assistant.invoke({"input": "What is the SLA for billing?"},
   config = cfg

)

Dropped 2 messages from history
Current messages: [HumanMessage(content='I have a billing issue with my last invoice.', additional_kwargs={}, response_metadata={}), AIMessage(content='Classification: Billing  \nPriority: High  \n\nI understand that you have a billing issue with your last invoice. Could you please provide more details about the problem? This will help me assist you better.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category is his complaint?', additional_kwargs={}, response_metadata={}), AIMessage(content='The complaint is categorized as:  \nClassification: Billing  \nPriority: High  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the SLA for billing?', additional_kwargs={}, response_metadata={}), AIMessage(content='The standard Service Level Agreement (SLA) for billing inquiries typically ranges from 24 to 48 hours for initial respons

In [66]:
output

'The standard Service Level Agreement (SLA) for billing inquiries typically ranges from 24 to 48 hours for initial response. However, response times can vary based on the complexity of the issue. If you have a specific issue you’d like assistance with, please let me know!'